<a href="https://colab.research.google.com/github/kimgumin20214253/net_guardian/blob/main/notebooks/net_guardian_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
`import numpy as np
import pandas as pd
import pickle
import time
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score
from lightgbm import LGBMClassifier

# ==========================================
# 1. [팀장 선언] 피처 순서 및 규격 강제 정의
# ==========================================
# 승현님의 feature_extractor.py가 도출할 통계 지표 순서를 미리 고정합니다.
FEATURE_COLUMNS = ['avg_rtt', 'max_rtt', 'std_rtt', 'moving_avg', 'rtt_change_rate']

print("[*] 1단계: 모델 학습용 피처 가이드라인 설정 완료")

# ==========================================
# 2. 가짜 데이터(Dummy Data) 생성 (구민님 데이터 대기용)
# ==========================================
# 실제 개발 시에는 이 부분을 구민님이 준 CSV를 읽어오는 코드로 바꿉니다.
# 예: df_normal = pd.read_csv("raw_normal.csv") ... 후 concat

print("[*] 2단계: 구민님 데이터 시뮬레이션을 위한 가짜 데이터 600건 생성 중...")
np.random.seed(42)

# 가상의 정상 데이터 300건 (RTT가 낮고 안정적임)
normal_data = np.random.uniform(10, 25, size=(300, 5))
df_normal = pd.DataFrame(normal_data, columns=FEATURE_COLUMNS)
df_normal['is_anomaly'] = 0

# 가상의 장애 데이터 300건 (RTT가 높고 변동성이 큼)
anomaly_data = np.random.uniform(40, 150, size=(300, 5))
df_anomaly = pd.DataFrame(anomaly_data, columns=FEATURE_COLUMNS)
df_anomaly['is_anomaly'] = 1

# 데이터 병합 (기획서상 500회 이상 데이터 수집 목표 반영)
df_total = pd.concat([df_normal, df_anomaly], ignore_index=True)

# 피처(X)와 라벨(y) 분리
X = df_total[FEATURE_COLUMNS]
y = df_total['is_anomaly']

# ==========================================
# 3. 5-Fold 교차 검증 및 하이퍼파라미터 튜닝
# ==========================================
print("[*] 3단계: 경진대회 배점 저격용 5-Fold 및 GridSearchCV 가동")

# 데이터 분할 방식 정의 (클래스 비율을 균일하게 유지하는 StratifiedKFold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# [모델 A] RandomForest 튜닝 후보 지정
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [4, 6, 8],
    'random_state': [42]
}
grid_rf = GridSearchCV(RandomForestClassifier(), rf_params, cv=cv, scoring='f1', n_jobs=-1)
grid_rf.fit(X, y)
print(f"  - RandomForest 최적 파라미터: {grid_rf.best_params_}")

# [모델 B] LightGBM 튜닝 후보 지정 (추론 속도 우수)
lgb_params = {
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'random_state': [42],
    'verbosity': [-1]
}
grid_lgb = GridSearchCV(LGBMClassifier(), lgb_params, cv=cv, scoring='f1', n_jobs=-1)
grid_lgb.fit(X, y)
print(f"  - LightGBM 최적 파라미터: {grid_lgb.best_params_}")

# ==========================================
# 4. Soft Voting 앙상블 결합 (최종 모델)
# ==========================================
print("[*] 4단계: 최적화된 모델들로 Soft Voting 앙상블 구축")

best_rf = grid_rf.best_estimator_
best_lgbm = grid_lgb.best_estimator_

final_ensemble = VotingClassifier(
    estimators=[('rf', best_rf), ('lgbm', best_lgbm)],
    voting='soft'  # 확률 기반 소프트 보팅으로 의사결정 결함률 최소화
)
final_ensemble.fit(X, y)

# ==========================================
# 5. 모델 성능 평가 및 서빙 파일(.pkl) 추출
# ==========================================
print("[*] 5단계: 최종 모델 검증 및 pkl 파일 추출")
y_pred = final_ensemble.predict(X)
accuracy = accuracy_score(y, y_pred)
print(f"  - 현재 Dummy 데이터 기준 탐지 정확도: {accuracy * 100:.2f}% (목표: 80% 이상)")

# 승현님 대시보드 인프라에 이식할 파일로 내보내기
with open("final_model.pkl", "wb") as f:
    pickle.dump(final_ensemble, f)
print("[+] 완료! 'final_model.pkl' 파일이 성공적으로 생성되었습니다.")


[*] 1단계: 모델 학습용 피처 가이드라인 설정 완료
[*] 2단계: 구민님 데이터 시뮬레이션을 위한 가짜 데이터 600건 생성 중...
[*] 3단계: 경진대회 배점 저격용 5-Fold 및 GridSearchCV 가동
  - RandomForest 최적 파라미터: {'max_depth': 4, 'n_estimators': 50, 'random_state': 42}
  - LightGBM 최적 파라미터: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50, 'random_state': 42, 'verbosity': -1}
[*] 4단계: 최적화된 모델들로 Soft Voting 앙상블 구축
[*] 5단계: 최종 모델 검증 및 pkl 파일 추출
  - 현재 Dummy 데이터 기준 탐지 정확도: 100.00% (목표: 80% 이상)
[+] 완료! 'final_model.pkl' 파일이 성공적으로 생성되었습니다.
